### Patient List Processing

In [3]:
import pandas as pd

# Path to your Excel file
file_path = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/TRAPPARDS20232025.xlsx'

# Read the specific sheet
df1 = pd.read_excel(file_path, sheet_name='PatientInfo')
df2 = pd.read_excel(file_path, sheet_name='EncounterProperties')

# Display the first few rows of the dataframe
print(df1.head())
print(df2.head())

                              PatientID        MRN               DOB
0  12288E12-375C-E311-800F-00215A9B0094   38759140  08/25/2007 00:00
1  43C395B6-395C-E311-800F-00215A9B0094  100117962  09/02/2010 00:00
2  D7970BC6-445C-E311-800F-00215A9B0094  100205702  03/16/2009 00:00
3  1912AB08-455C-E311-800F-00215A9B0094  100151965  04/12/2000 00:00
4  1E4D6298-455C-E311-800F-00215A9B0094   41994058  09/11/2007 00:00
                              PatientID                           EncounterID  \
0  12288E12-375C-E311-800F-00215A9B0094  D5A16D53-438B-EF11-A1D0-0025B5000031   
1  12288E12-375C-E311-800F-00215A9B0094  DEBC5252-4F8B-EF11-A1D0-0025B5000031   
2  12288E12-375C-E311-800F-00215A9B0094  D8AEAB0D-BB8B-EF11-A1D0-0025B5000031   
3  12288E12-375C-E311-800F-00215A9B0094  7382981F-BB8B-EF11-A1D0-0025B5000031   
4  12288E12-375C-E311-800F-00215A9B0094  7882981F-BB8B-EF11-A1D0-0025B5000031   

          AdmitDate     DischargeDate  ICUStayYN  ICUDays  ICUMinutes  \
0  10/15/2024 17:38  10/24

In [4]:
df1.shape

(634, 3)

In [5]:
len(df1['PatientID'].unique())

634

In [6]:
len(df1['MRN'].unique())

634

In [7]:
df2.shape

(55071, 10)

In [8]:
len(df2['PatientID'].unique())

634

In [9]:
# Merge the dataframes on the "PatientID" column
merged_df = pd.merge(df1, df2, on='PatientID')

# Format the 'MRN' column to have exactly 9 digits with leading zeros
merged_df['MRN'] = merged_df['MRN'].astype(str).str.zfill(9)

# Display the first few rows of the merged dataframe
print(merged_df.head())


                              PatientID        MRN               DOB  \
0  12288E12-375C-E311-800F-00215A9B0094  038759140  08/25/2007 00:00   
1  12288E12-375C-E311-800F-00215A9B0094  038759140  08/25/2007 00:00   
2  12288E12-375C-E311-800F-00215A9B0094  038759140  08/25/2007 00:00   
3  12288E12-375C-E311-800F-00215A9B0094  038759140  08/25/2007 00:00   
4  12288E12-375C-E311-800F-00215A9B0094  038759140  08/25/2007 00:00   

                            EncounterID         AdmitDate     DischargeDate  \
0  D5A16D53-438B-EF11-A1D0-0025B5000031  10/15/2024 17:38  10/24/2024 13:27   
1  DEBC5252-4F8B-EF11-A1D0-0025B5000031  10/15/2024 19:45  10/15/2024 19:45   
2  D8AEAB0D-BB8B-EF11-A1D0-0025B5000031  10/15/2024 17:41  10/15/2024 17:41   
3  7382981F-BB8B-EF11-A1D0-0025B5000031  10/15/2024 17:29  10/15/2024 17:29   
4  7882981F-BB8B-EF11-A1D0-0025B5000031  10/15/2024 12:34  10/15/2024 12:56   

   ICUStayYN  ICUDays  ICUMinutes HospitalAdmissionDate HospitalDischargeDate  \
0        1.

In [10]:
merged_df.shape

(55071, 12)

In [11]:
len(merged_df['MRN'].unique())

634

In [12]:
len(merged_df['PatientID'].unique())

634

In [13]:
# Keep only the unique combinations of 'PatientID' and 'HospitalAdmissionDate'
merged_df_sub = merged_df.drop_duplicates(subset=['PatientID', 'HospitalAdmissionDate'])

# Group by 'PatientID' and fill NaNs in 'HospitalAdmissionDate' with the corresponding 'AdmitDate'
def fill_hospital_admission_date(group):
    if group['HospitalAdmissionDate'].isna().all():
        group['HospitalAdmissionDate'] = group['AdmitDate']
    return group

merged_df_sub = merged_df_sub.groupby('PatientID').apply(fill_hospital_admission_date).reset_index(drop=True)
merged_df_sub.shape

(2099, 12)

In [14]:
len(merged_df_sub['MRN'].unique())

634

In [15]:
len(merged_df_sub['PatientID'].unique())

634

In [16]:
# Drop rows where 'HospitalAdmissionDate' is duplicated
merged_df_sub_sub = merged_df_sub.drop_duplicates(subset=['PatientID', 'HospitalAdmissionDate'])

# Drop rows where 'HospitalAdmissionDate' is NaN
merged_df_sub_sub = merged_df_sub_sub.dropna(subset=['HospitalAdmissionDate'])

merged_df_sub_sub.shape

(1466, 12)

In [17]:
len(merged_df_sub_sub['MRN'].unique())

634

In [18]:
len(merged_df_sub_sub['PatientID'].unique())

634

In [19]:
df = merged_df_sub_sub
df.shape

(1466, 12)

In [20]:
import pandas as pd

# Convert 'HospitalAdmissionDate' and 'DOB' to datetime if they are not already
df['HospitalAdmissionDate'] = pd.to_datetime(df['HospitalAdmissionDate'])
df['DOB'] = pd.to_datetime(df['DOB'])

# Calculate the age in years
df['Age'] = (df['HospitalAdmissionDate'] - df['DOB']).dt.total_seconds() / (365.25 * 24 * 60 * 60)

# Drop the specified columns
df_sub = df.drop(columns=['PatientID', 'EncounterID', 'ICUMinutes', 'AdmitDate', 'DischargeDate'])

# Reorder and rename the columns
new_column_order = ['MRN', 'DOB', 'Age', 'Weight', 'HospitalAdmissionDate', 
                    'HospitalDischargeDate', 'ICUStayYN', 'ICUDays']
df_sub = df_sub[new_column_order]

df_sub

,MRN,DOB,Age,Weight,HospitalAdmissionDate,HospitalDischargeDate,ICUStayYN,ICUDays
0,038759140,2007-08-25,17.143695,3280.44,2024-10-15 17:38:00,10/24/2024 13:27,1.0,6.0
2,100117962,2010-09-02,12.793102,1329.81,2023-06-18 16:20:00,06/21/2023 12:35,1.0,1.0
5,100205702,2009-03-16,14.804356,1477.96,2024-01-04 06:59:00,02/19/2024 15:19,1.0,12.0
7,100151965,2000-04-12,24.211834,2804.25,2024-06-28 08:56:00,07/13/2024 18:14,1.0,1.0
8,100151965,2000-04-12,24.242300,NaN,2024-07-09 12:00:00,07/09/2024 23:59,0.0,NaN
...,...,...,...,...,...,...,...,...
2092,100574112,2015-06-03,8.582181,NaN,2024-01-01 15:24:00,01/01/2024 23:59,0.0,NaN
2093,100574112,2015-06-03,8.666419,NaN,2024-02-01 09:50:00,02/01/2024 23:59,0.0,NaN
2095,100478650,2015-06-27,9.419397,NaN,2024-11-26 10:26:00,11/26/2024 23:59,0.0,NaN
2096,100478650,2015-06-27,9.419313,843.04,2024-11-26 09:42:00,11/30/2024 09:45,1.0,4.0


In [21]:
len(df_sub['MRN'].unique())

634

In [22]:
# Ensure 'HospitalAdmissionDate' is in datetime format
df_sub['HospitalAdmissionDate_Date'] = pd.to_datetime(df_sub['HospitalAdmissionDate'])

# Check the minimum date
min_date = df_sub['HospitalAdmissionDate_Date'].min()

# Check the maximum date
max_date = df_sub['HospitalAdmissionDate_Date'].max()

# Display the results
print(f"Minimum Hospital Admission Date: {min_date}")
print(f"Maximum Hospital Admission Date: {max_date}")

Minimum Hospital Admission Date: 2022-08-11 01:12:00
Maximum Hospital Admission Date: 2025-02-17 11:57:00


In [23]:
import pandas as pd

df_sub.to_csv('/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_V2_df.csv', index=False)

print("DataFrame saved to 'PARDS_Risk_V2_df.csv'")

DataFrame saved to 'PARDS_Risk_V2_df.csv'


### Identify the Time Start for each event.

In [2]:
import pandas as pd

df_sub = pd.read_csv('/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_V2_df.csv')

sample_events_MRN = df_sub[['MRN', 'HospitalAdmissionDate']]

# Rename the MRN and HospitalAdmissionDate columns to mrn and Time Start
sample_events_MRN = sample_events_MRN.copy()
sample_events_MRN.rename(columns={'MRN': 'mrn', 'HospitalAdmissionDate': 'Time Start'}, inplace=True)

# Format the 'mrn' column to have exactly 9 digits with leading zeros
sample_events_MRN['mrn'] = sample_events_MRN['mrn'].astype(str).str.zfill(9)

sample_events_MRN

,mrn,Time Start
0,038759140,10/15/24 17:38
1,100117962,6/18/23 16:20
2,100205702,1/4/24 6:59
3,100151965,6/28/24 8:56
4,100151965,7/9/24 12:00
...,...,...
1461,100574112,1/1/24 15:24
1462,100574112,2/1/24 9:50
1463,100478650,11/26/24 10:26
1464,100478650,11/26/24 9:42


In [3]:
len(sample_events_MRN['mrn'].unique())

634

In [5]:
import pandas as pd

# Ensure 'Time Start' column is in datetime format
sample_events_MRN['Time Start'] = pd.to_datetime(sample_events_MRN['Time Start'])

# Define the date correctly
july_2024 = pd.Timestamp("2024-07-21")

# Filter the dataframe
sample_events_MRN_sub = sample_events_MRN[sample_events_MRN['Time Start'] >= july_2024]

sample_events_MRN_sub

,mrn,Time Start
0,038759140,2024-10-15 17:38:00
11,100238080,2024-08-19 16:48:00
12,100238080,2024-08-19 12:57:00
16,100237060,2024-11-27 20:17:00
17,100203261,2024-11-11 06:00:00
...,...,...
1451,100539560,2024-10-21 21:31:00
1453,100567428,2024-12-28 11:57:00
1463,100478650,2024-11-26 10:26:00
1464,100478650,2024-11-26 09:42:00


In [6]:
len(sample_events_MRN_sub['mrn'].unique())

225

#### 2 hours before hospital admission date & 10 hours after hospital admission date.

In [4]:
import pandas as pd

result_df = sample_events_MRN.copy()

result_df['Time Stop'] = result_df['Time Start']

# Convert 'Time Start' and 'Time Stop' to datetime format
result_df['Time Start'] = pd.to_datetime(result_df['Time Start'])
result_df['Time Stop'] = pd.to_datetime(result_df['Time Stop'])

# Create new 'Time Start' as 'Time Stop' minus 2 hours
result_df['Time Start'] = result_df['Time Stop'] - pd.to_timedelta('2:00:00')

# Create new 'Time Stop' as 'Time Start' plus 18 hours
result_df['Time Stop'] = result_df['Time Start'] + pd.to_timedelta('12:00:00')

# Format the 'mrn' column to have exactly 9 digits with leading zeros
result_df['mrn'] = result_df['mrn'].astype(str).str.zfill(9)

# Save the DataFrame as a new Excel file
result_df.to_excel('/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/sample_events_MRN_2hrsBeforeHAD_10hrsAfterHAD.xlsx', index=False)

print("DataFrame saved as 'sample_events_MRN_2hrsBeforeHAD_10hrsAfterHAD.xlsx'")

DataFrame saved as 'sample_events_MRN_2hrsBeforeHAD_10hrsAfterHAD.xlsx'


### File Names Regulation

In [23]:
# Add row information to the event list file
import pandas as pd

# Define the file path
# ## 1 to 200
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_1_to_200/Study23_Tag210_EventList.csv"

# ## 200 to 400
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_200_to_400/Study23_Tag207_EventList.csv"

# ## 400 to 600
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_400_to_600/Study23_Tag200_EventList.csv"
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_400_to_600/Study23_Tag200_EventList(1to129).csv"
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_400_to_600/Study23_Tag200_EventList(130to200).csv"

## 600 to 800
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_600_to_800/Study23_Tag201_EventList.csv"
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_600_to_800/Study23_Tag201_EventList(1to175).csv"
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_600_to_800/Study23_Tag201_EventList(176to200).csv"

# ## 800 to 1000
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_800_to_1000/Study23_Tag208_EventList.csv"

# ## 1000 to 1200
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_1000_to_1200/Study23_Tag203_EventList.csv"

# ## 1200 to 1466
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_1200_to_1466/Study23_Tag204_EventList.csv"

# ## PARDS_Risk_V1_2nd_hrs_Raw
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/Study23_Tag167_EventList.csv"

# ## PARDS_Risk_V1_3rd_hrs_Raw
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/PARDS_Risk_V1_3rd_Hour_Raw/Study23_Tag221_EventList.csv"

# ## PARDS_Risk_V1_13th_hrs_Raw
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/Study23_Tag168_EventList.csv"

# ## PARDS_Risk_V1_14th_hrs_Raw
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/PARDS_Risk_V1_14th_Hour_Raw/Study23_Tag222_EventList.csv"

# ## PARDS_Risk_V2_1st_hrs_Raw
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/1st_Hour_Raw/Study23_Tag212_EventList.csv"

# ## PARDS_Risk_V2_2nd_hrs_Raw
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/2nd_Hour_Raw/Study23_Tag213_EventList.csv"

# ## PARDS_Risk_V2_3rd_hrs_Raw
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/3rd_Hour_Raw/Study23_Tag214_EventList.csv"

# ## PARDS_Risk_V2_12th_hrs_Raw
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/12th_Hour_Raw/Study23_Tag217_EventList.csv"

# ## PARDS_Risk_V2_13th_hrs_Raw
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/13th_Hour_Raw/Study23_Tag218_EventList.csv"

# ## PARDS_Risk_V2_14th_hrs_Raw
# file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/14th_Hour_Raw/Study23_Tag219_EventList.csv"

# Read the CSV file
df = pd.read_csv(file_path)

# Create a new column for row numbers
df.insert(0, 'Row Number', [f"Row_{i+1}" for i in range(len(df))])

# Save the modified DataFrame back to the same path
df.to_csv(file_path, index=False)

# Display the modified DataFrame
print(df.head())


  Row Number   Mark  Patient ID  Tag ID study_name  study_id  \
0      Row_1  38476        1288     219  PARDS_LWT        23   
1      Row_2  38514        1748     219  PARDS_LWT        23   
2      Row_3  38685        1748     219  PARDS_LWT        23   
3      Row_4  38574        2294     219  PARDS_LWT        23   
4      Row_5  38706        7028     219  PARDS_LWT        23   

                  tag_name          Time Start UTC           Time Stop UTC  \
0  PARDS_Risk_V2_14th_Hour  2023-12-02 06:47:45+00  2023-12-02 07:47:45+00   
1  PARDS_Risk_V2_14th_Hour  2024-02-21 21:44:01+00  2024-02-21 22:44:01+00   
2  PARDS_Risk_V2_14th_Hour  2024-11-05 19:42:01+00  2024-11-05 20:42:01+00   
3  PARDS_Risk_V2_14th_Hour  2024-05-21 11:22:37+00  2024-05-21 12:22:37+00   
4  PARDS_Risk_V2_14th_Hour  2024-12-13 07:56:18+00  2024-12-13 08:56:18+00   

            Time Start            Time Stop  \
0  2023-12-02 01:47:45  2023-12-02 02:47:45   
1  2024-02-21 16:44:01  2024-02-21 17:44:01   
2  20

In [32]:
import pandas as pd
import os
import glob
import shutil
import re

# Define the file paths
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_1_to_200/Study23_Tag210_EventList"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_200_to_400/Study23_Tag207_EventList"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_400_to_600/Study23_Tag200_EventList(1to129)"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_400_to_600/Study23_Tag200_EventList(130to200)"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_600_to_800/Study23_Tag201_EventList(1to175)"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_600_to_800/Study23_Tag201_EventList(176to200)"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_800_to_1000/Study23_Tag208_EventList"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_1000_to_1200/Study23_Tag203_EventList"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_1200_to_1466/Study23_Tag204_EventList"

# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/PARDS_RIsk_V1_2nd_hrs_Raw"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/PARDS_Risk_V1_3rd_Hour_Raw/Study23_Tag221_EventList"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/PARDS_Risk_V1_13th_hrs_Raw"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/PARDS_Risk_V1_14th_Hour_Raw/Study23_Tag222_EventList"

# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/1st_Hour_Raw/Study23_Tag212_EventList"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/2nd_Hour_Raw/Study23_Tag213_EventList"
directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/3rd_Hour_Raw/Study23_Tag214_EventList"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/12th_Hour_Raw/Study23_Tag217_EventList"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/13th_Hour_Raw/Study23_Tag218_EventList"
# directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/14th_Hour_Raw/Study23_Tag219_EventList"

renamed_directory = os.path.join(directory_path, "Renamed")

# Ensure the renamed directory exists
os.makedirs(renamed_directory, exist_ok=True)

# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_1_to_200/Study23_Tag210_EventList.csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_200_to_400/Study23_Tag207_EventList.csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_400_to_600/Study23_Tag200_EventList(1to129).csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_400_to_600/Study23_Tag200_EventList(130to200).csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_600_to_800/Study23_Tag201_EventList(1to175).csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_600_to_800/Study23_Tag201_EventList(176to200).csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_800_to_1000/Study23_Tag208_EventList.csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_1000_to_1200/Study23_Tag203_EventList.csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Raw_2hrsBeforeHAD_12hrsAfterHAD_1200_to_1466/Study23_Tag204_EventList.csv"

# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/Study23_Tag167_EventList.csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/PARDS_Risk_V1_3rd_Hour_Raw/Study23_Tag221_EventList.csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/Study23_Tag168_EventList.csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/PARDS_Risk_V1_14th_Hour_Raw/Study23_Tag222_EventList.csv"

# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/1st_Hour_Raw/Study23_Tag212_EventList.csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/2nd_Hour_Raw/Study23_Tag213_EventList.csv"
event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/3rd_Hour_Raw/Study23_Tag214_EventList.csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/12th_Hour_Raw/Study23_Tag217_EventList.csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/13th_Hour_Raw/Study23_Tag218_EventList.csv"
# event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/14th_Hour_Raw/Study23_Tag219_EventList.csv"

# Read the event list CSV
event_df = pd.read_csv(event_list_path, dtype={"Row Number": str})

# Create a mapping from Row Number to PatientID and StartTime
event_mapping = {str(row["Row Number"]): (row["Patient ID"], row["Time Start"]) for _, row in event_df.iterrows()}

# Get all CSV files in the directory
csv_files = glob.glob(os.path.join(directory_path, "*.csv"))

for file_path in csv_files:
    file_name = os.path.basename(file_path)
    
    # Extract the row number from the filename using regex
    match = re.search(r"Row_\d+", file_name)
    if match:
        row_key = match.group()
        
        # Check if extracted row number is in the mapping
        if row_key in event_mapping:
            patient_id, start_time = event_mapping[row_key]
            
            # Convert StartTime to required format
            try:
                formatted_time = pd.to_datetime(start_time).strftime("%Y%m%d_%H")
                cat_name = "V2_3rd_Raw"
                new_file_name = f"{patient_id}_{formatted_time}_{cat_name}.csv"
                new_file_path = os.path.join(renamed_directory, new_file_name)
                
                # Copy the file instead of moving it
                shutil.copy2(file_path, new_file_path)
                print(f"Copied and renamed: {file_name} -> {new_file_path}")
            except Exception as e:
                print(f"Error processing file {file_name}: {e}")


Copied and renamed: Event_Row_229_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/3rd_Hour_Raw/Study23_Tag214_EventList/Renamed/1362933_20240826_13_V2_3rd_Raw.csv
Copied and renamed: Event_Row_169_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/3rd_Hour_Raw/Study23_Tag214_EventList/Renamed/1117361_20240306_05_V2_3rd_Raw.csv
Copied and renamed: Event_Row_250_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/3rd_Hour_Raw/Study23_Tag214_EventList/Renamed/1454681_20241105_05_V2_3rd_Raw.csv
Copied and renamed: Event_Row_258_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/3rd_Hour_Raw/Study23_Tag214_EventList/Renamed/1484434_20250128_19_V2_3rd_Raw.csv
Copied and renamed: Event_Row_66_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/3rd_Hou

### Create a information table

In [20]:
import pandas as pd

# Define file path
file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/user_event_check_tag_194.csv"

# Read the CSV file
df = pd.read_csv(file_path)

# Ensure required columns exist
if "Patient ID" in df.columns and "Time Start" in df.columns:
    # Convert 'Time Start' to datetime format
    df["Time Start"] = pd.to_datetime(df["Time Start"], errors='coerce')

    # Function to generate file name, handling NaT values
    def generate_filename(row):
        if pd.isna(row["Time Start"]):
            return f"{row['Patient ID']}_InvalidDate"
        return f"{row['Patient ID']}_{row['Time Start'].strftime('%Y%m%d_%H')}"

    # Apply the function to create the new column
    df["Raw_File_Name"] = df.apply(generate_filename, axis=1)

    # Save the updated DataFrame
    output_path = file_path.replace(".csv", "_updated.csv")
    df.to_csv(output_path, index=False)
    print(f"File saved successfully at: {output_path}")
else:
    print("Error: Required columns 'Patient ID' or 'Time Start' not found in the CSV file.")


File saved successfully at: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/user_event_check_tag_194_updated.csv


In [27]:
import pandas as pd

# Define file paths
source_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_V2_df.csv"
target_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/user_event_check_tag_194_updated.csv"
output_xlsx = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_V2_df.xlsx"

# Read the source file with additional columns
df_source = pd.read_csv(source_file, dtype={"MRN": str})

# Read the target file
df_target = pd.read_csv(target_file, dtype={"MRN": str})

# Ensure 'MRN' column exists in both dataframes
if "MRN" in df_source.columns and "MRN" in df_target.columns:
    # Ensure MRN is 9 digits by padding with zeros if necessary
    df_source["MRN"] = df_source["MRN"].str.zfill(9)
    df_target["MRN"] = df_target["MRN"].str.zfill(9)
    
    # Select only relevant columns from source file and drop duplicates
    cols_to_merge = ["MRN", "DOB", "Age", "Weight", "HospitalAdmissionDate", "HospitalDischargeDate", "ICUStayYN", "ICUDays"]
    df_source_unique = df_source[cols_to_merge].drop_duplicates(subset=["MRN"])
    
    # Convert date columns to match 'Time Start' and 'Time Stop' format
    date_columns = ["DOB", "HospitalAdmissionDate", "HospitalDischargeDate"]
    for col in date_columns:
        if col in df_source_unique.columns:
            df_source_unique[col] = pd.to_datetime(df_source_unique[col], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
    
    # Loop through target file and add the matching information
    df_target = df_target.merge(df_source_unique, on="MRN", how="left")
    
    # Save the updated DataFrame as an Excel file
    df_target.to_excel(output_xlsx, index=False)
    print(f"File updated successfully and saved at: {output_xlsx}")
else:
    print("Error: 'MRN' column not found in one or both files.")


File updated successfully and saved at: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_V2_df.xlsx


### Identify the starting point of each event & Change the row names in each event to signal names.

In [34]:
# Demo
import pandas as pd
import os

# Define file paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_V2_df.xlsx"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/0_Raw_2hrsBeforeHAD_12hrsAfterHAD_1_to_1466"
signal_info_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Study23_Signal_Row_Info.csv"
demo_folder = os.path.join(csv_folder, "Demo_RenameRowNames")

# Create folder for demo processed files if it doesn't exist
os.makedirs(demo_folder, exist_ok=True)

# Read the Excel file
df_xlsx = pd.read_excel(xlsx_file, dtype={"MRN": str})

# Read signal info file
df_signal_info = pd.read_csv(signal_info_file)
signal_map = dict(zip(df_signal_info["Signal Name"], df_signal_info["Display Name"]))

# Define ventilation type signal groups
# vent_sets = {
#     "AVEA": ["CAPSULE_AVEAA_VITAL_2564", "CAPSULE_AVEAA_VITAL_1189", "CAPSULE_AVEAA_VITAL_2325", "CAPSULE_AVEAA_WAVE_52911"],
#     "CDGR": ["CAPSULE_DRAGERMEDIBUS_VITAL_2564", "CAPSULE_DRAGERMEDIBUS_VITAL_1189", "CAPSULE_DRAGERMEDIBUS_VITAL_2325", "CAPSULE_DRAGERMEDIBUS_WAVE_118"],
#     "SVU": ["CAPSULE_MAQUETC_VITAL_1414", "CAPSULE_MAQUETC_VITAL_1189", "CAPSULE_MAQUETC_VITAL_2325", "MAQUET_CAPSULE_AIRWAY_FLOW_WAVE"]
# }

vent_sets = {
    "AVEA": ["AVEA - PIP", "AVEA - PEEP", "AVEA - eVT", "AVEA - Air Flow Wave"],
    "CDGR": ["CDGR - PIP", "CDGR - PEEP", "CDGR - eVT", "CDGR - Flow"],
    "SVU": ["SVU - PIP", "SVU - PEEP", "SVU - eVT", "SVU - Flow"]
}

# Get the first CSV file in the folder
csv_files = [f for f in os.listdir(csv_folder) if f.endswith(".csv")]

if csv_files:
    csv_file = csv_files[0]  # Take the first CSV file
    file_name = csv_file.replace(".csv", "")

    # Find matching row in Excel file
    matched_row = df_xlsx[df_xlsx["Raw_File_Name"] == file_name]

    if not matched_row.empty:
        csv_path = os.path.join(csv_folder, csv_file)
        df_csv = pd.read_csv(csv_path)

        # Rename columns based on signal mapping
        df_csv.rename(columns=signal_map, inplace=True)

        # Identify 1st_Time_Start as the first occurrence where one set of signals is non-empty
        df_valid = df_csv.dropna(subset=["Time"])
        first_time_start = None
        vent_type = "Unknown"

        for idx, row in df_valid.iterrows():
            for vent, signals in vent_sets.items():
                if all(sig in df_valid.columns and pd.notna(row[sig]) for sig in signals):
                    first_time_start = row["Time"]
                    vent_type = vent
                    break
            if first_time_start:
                break

        # Print results for debugging
        print(f"Processing file: {csv_file}")
        print(f"1st_Time_Start: {first_time_start}")
        print(f"Ventilation Type: {vent_type}")

        # Add column for ventilation type
        df_csv["Vent_Type"] = vent_type

        # Save updated CSV for demo
        demo_csv_path = os.path.join(demo_folder, f"demo_{csv_file}")
        df_csv.to_csv(demo_csv_path, index=False)
        print(f"Demo file saved: {demo_csv_path}")
else:
    print("No CSV files found in the folder.")


Processing file: 1301993_20240830_15.csv
1st_Time_Start: 2024-08-30 20:22:02.000
Ventilation Type: CDGR
Demo file saved: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/0_Raw_2hrsBeforeHAD_12hrsAfterHAD_1_to_1466/Demo_RenameRowNames/demo_1301993_20240830_15.csv


In [ ]:
# All
import pandas as pd
import os

# Define file paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_V2_df.xlsx"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/0_Raw_2hrsBeforeHAD_12hrsAfterHAD_1_to_1466"
signal_info_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/Study23_Signal_Row_Info.csv"
rename_folder = os.path.join(csv_folder, "RenameRowNames")

# Create folder for processed files if it doesn't exist
os.makedirs(rename_folder, exist_ok=True)

# Read the Excel file
df_xlsx = pd.read_excel(xlsx_file, dtype={"MRN": str})

# Read signal info file
df_signal_info = pd.read_csv(signal_info_file)
signal_map = dict(zip(df_signal_info["Signal Name"], df_signal_info["Display Name"]))

# Define ventilation type signal groups
# vent_sets = {
#     "AVEA": ["CAPSULE_AVEAA_VITAL_2564", "CAPSULE_AVEAA_VITAL_1189", "CAPSULE_AVEAA_VITAL_2325", "CAPSULE_AVEAA_WAVE_52911"],
#     "CDGR": ["CAPSULE_DRAGERMEDIBUS_VITAL_2564", "CAPSULE_DRAGERMEDIBUS_VITAL_1189", "CAPSULE_DRAGERMEDIBUS_VITAL_2325", "CAPSULE_DRAGERMEDIBUS_WAVE_118"],
#     "SVU": ["CAPSULE_MAQUETC_VITAL_1414", "CAPSULE_MAQUETC_VITAL_1189", "CAPSULE_MAQUETC_VITAL_2325", "MAQUET_CAPSULE_AIRWAY_FLOW_WAVE"]
# }

vent_sets = {
    "AVEA": ["AVEA - PIP", "AVEA - PEEP", "AVEA - eVT", "AVEA - Air Flow Wave"],
    "CDGR": ["CDGR - PIP", "CDGR - PEEP", "CDGR - eVT", "CDGR - Flow"],
    "SVU": ["SVU - PIP", "SVU - PEEP", "SVU - eVT", "SVU - Flow"]
}

# Create dictionaries to store results
first_time_start_dict = {}
vent_type_dict = {}

# Loop through each CSV file in the folder
for csv_file in os.listdir(csv_folder):
    if csv_file.endswith(".csv"):
        file_name = csv_file.replace(".csv", "")
        
        # Find matching row in Excel file
        matched_row = df_xlsx[df_xlsx["Raw_File_Name"] == file_name]
        
        if not matched_row.empty:
            csv_path = os.path.join(csv_folder, csv_file)
            df_csv = pd.read_csv(csv_path)
            
            # Rename columns based on signal mapping
            df_csv.rename(columns=signal_map, inplace=True)
            
            # Identify 1st_Time_Start as the first occurrence where one set of signals is non-empty
            df_valid = df_csv.dropna(subset=["Time"])
            first_time_start = None
            
            for idx, row in df_valid.iterrows():
                for vent, signals in vent_sets.items():
                    if all(sig in df_valid.columns and pd.notna(row[sig]) for sig in signals):
                        first_time_start = row["Time"]
                        vent_type = vent
                        break
                if first_time_start:
                    break
            
            # Store results in dictionaries
            first_time_start_dict[file_name] = first_time_start
            vent_type_dict[file_name] = vent_type if first_time_start else "Unknown"
            
            # Add column for ventilation type
            df_csv["Vent_Type"] = vent_type_dict[file_name]
            
            # Save updated CSV in new folder
            processed_csv_path = os.path.join(rename_folder, csv_file)
            df_csv.to_csv(processed_csv_path, index=False)
            print(f"Processed {csv_file}: Vent_Type={vent_type_dict[file_name]}, 1st_Time_Start={first_time_start}")

# Add new columns to the Excel file
df_xlsx["1st_Time_Start"] = df_xlsx["Raw_File_Name"].map(first_time_start_dict)
df_xlsx["Vent_Type"] = df_xlsx["Raw_File_Name"].map(vent_type_dict)
df_xlsx.to_excel(xlsx_file, index=False)
print(f"Updated {xlsx_file} with 1st_Time_Start and Vent_Type columns.")


### Identify the Starting Point Only

In [1]:
# Demo
import pandas as pd
import os

# Define file paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_V2_df.xlsx"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/0_Raw_2hrsBeforeHAD_12hrsAfterHAD_1_to_1466"

# Read the Excel file
df_xlsx = pd.read_excel(xlsx_file, dtype={"MRN": str})

# Define ventilation type signal groups
vent_sets = {
    "AVEA": ["CAPSULE_AVEAA_VITAL_2564", "CAPSULE_AVEAA_VITAL_1189", "CAPSULE_AVEAA_VITAL_2325", "CAPSULE_AVEAA_WAVE_52911"],
    "CDGR": ["CAPSULE_DRAGERMEDIBUS_VITAL_2564", "CAPSULE_DRAGERMEDIBUS_VITAL_1189", "CAPSULE_DRAGERMEDIBUS_VITAL_2325", "CAPSULE_DRAGERMEDIBUS_WAVE_118"],
    "SVU": ["CAPSULE_MAQUETC_VITAL_1414", "CAPSULE_MAQUETC_VITAL_1189", "CAPSULE_MAQUETC_VITAL_2325", "MAQUET_CAPSULE_AIRWAY_FLOW_WAVE"]
}

# Get the first CSV file in the folder
csv_files = [f for f in os.listdir(csv_folder) if f.endswith(".csv")]

if csv_files:
    csv_file = csv_files[0]  # Take the first CSV file
    file_name = csv_file.replace(".csv", "")
    
    # Find matching row in Excel file
    matched_row = df_xlsx[df_xlsx["Raw_File_Name"] == file_name]
    
    if not matched_row.empty:
        csv_path = os.path.join(csv_folder, csv_file)
        df_csv = pd.read_csv(csv_path)
        
        # Identify 1st_Time_Start as the first occurrence where one set of signals is non-empty
        df_valid = df_csv.dropna(subset=["Time"])
        first_time_start = None
        vent_type = "Unknown"
        
        for idx, row in df_valid.iterrows():
            for vent, signals in vent_sets.items():
                if all(sig in df_valid.columns and pd.notna(row[sig]) for sig in signals):
                    first_time_start = row["Time"]
                    vent_type = vent
                    break
            if first_time_start:
                break
        
        # Print results for debugging
        print(f"Processing file: {csv_file}")
        print(f"1st_Time_Start: {first_time_start}")
        print(f"Ventilation Type: {vent_type}")
else:
    print("No CSV files found in the folder.")


Processing file: 1301993_20240830_15.csv
1st_Time_Start: 2024-08-30 20:22:02.000
Ventilation Type: CDGR


In [1]:
# All
import pandas as pd
import os

# Define file paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_V2_df.xlsx"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/Raw_2hrsBeforeHAD_12hrsAfterHAD/0_Raw_2hrsBeforeHAD_12hrsAfterHAD_1_to_1466"

# Read the Excel file
df_xlsx = pd.read_excel(xlsx_file, dtype={"MRN": str})

# Define ventilation type signal groups
vent_sets = {
    "AVEA": ["CAPSULE_AVEAA_VITAL_2564", "CAPSULE_AVEAA_VITAL_1189", "CAPSULE_AVEAA_VITAL_2325", "CAPSULE_AVEAA_WAVE_52911"],
    "CDGR": ["CAPSULE_DRAGERMEDIBUS_VITAL_2564", "CAPSULE_DRAGERMEDIBUS_VITAL_1189", "CAPSULE_DRAGERMEDIBUS_VITAL_2325", "CAPSULE_DRAGERMEDIBUS_WAVE_118"],
    "SVU": ["CAPSULE_MAQUETC_VITAL_1414", "CAPSULE_MAQUETC_VITAL_1189", "CAPSULE_MAQUETC_VITAL_2325", "MAQUET_CAPSULE_AIRWAY_FLOW_WAVE"]
}

# Create dictionaries to store results
first_time_start_dict = {}
vent_type_dict = {}

# Loop through each CSV file in the folder
for csv_file in os.listdir(csv_folder):
    if csv_file.endswith(".csv"):
        file_name = csv_file.replace(".csv", "")
        
        # Find matching row in Excel file
        matched_row = df_xlsx[df_xlsx["Raw_File_Name"] == file_name]
        
        if not matched_row.empty:
            csv_path = os.path.join(csv_folder, csv_file)
            df_csv = pd.read_csv(csv_path)
            
            # Identify 1st_Time_Start as the first occurrence where one set of signals is non-empty
            df_valid = df_csv.dropna(subset=["Time"])
            first_time_start = None
            vent_type = "Unknown"
            
            for idx, row in df_valid.iterrows():
                for vent, signals in vent_sets.items():
                    if all(sig in df_valid.columns and pd.notna(row[sig]) for sig in signals):
                        first_time_start = row["Time"]
                        vent_type = vent
                        break
                if first_time_start:
                    break
            
            # Store results in dictionaries
            first_time_start_dict[file_name] = first_time_start
            vent_type_dict[file_name] = vent_type if first_time_start else "Unknown"
            
            print(f"Processed {csv_file}: Vent_Type={vent_type_dict[file_name]}, 1st_Time_Start={first_time_start}")

# Add new columns to the Excel file
df_xlsx["1st_Time_Start"] = df_xlsx["Raw_File_Name"].map(first_time_start_dict)
df_xlsx["Vent_Type"] = df_xlsx["Raw_File_Name"].map(vent_type_dict)
df_xlsx.to_excel(xlsx_file, index=False)
print(f"Updated {xlsx_file} with 1st_Time_Start and Vent_Type columns.")

####################################################

### PARDS Risk V1 (2nd / 13th, 3rd / 14th) merged to df_combined, and save as PARDS_Risk_V1_df

In [5]:
import pandas as pd

# File paths
source_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/df_combined.xlsx"
output_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V1_df.xlsx"

# Read the original Sheet18
df = pd.read_excel(source_file, sheet_name="Sheet18")

# Format MRN as 9-digit string
df["MRN"] = df["MRN"].astype(str).str.zfill(9)

# Ensure datetime format
df["1st_Time_Start"] = pd.to_datetime(df["1st_Time_Start"], errors='coerce')
df["12th_Time_Start"] = pd.to_datetime(df["12th_Time_Start"], errors='coerce')

# Add additional time columns
df["2nd_Time_Start"] = df["1st_Time_Start"] + pd.Timedelta(hours=1)
df["3rd_Time_Start"] = df["1st_Time_Start"] + pd.Timedelta(hours=2)
df["13th_Time_Start"] = df["12th_Time_Start"] + pd.Timedelta(hours=1)
df["14th_Time_Start"] = df["12th_Time_Start"] + pd.Timedelta(hours=2)

# Format timestamps
def format_time(ts):
    return ts.strftime("%Y%m%d_%H") if pd.notnull(ts) else ""

# Add new file name columns
df["FileName_V1_2nd"] = df.apply(lambda row: f"{row['PID']}_{format_time(row['2nd_Time_Start'])}_V1_2nd_Raw", axis=1)
df["FileName_V1_3rd"] = df.apply(lambda row: f"{row['PID']}_{format_time(row['3rd_Time_Start'])}_V1_3rd_Raw", axis=1)
df["FileName_V1_13th"] = df.apply(lambda row: f"{row['PID']}_{format_time(row['13th_Time_Start'])}_V1_13th_Raw", axis=1)
df["FileName_V1_14th"] = df.apply(lambda row: f"{row['PID']}_{format_time(row['14th_Time_Start'])}_V1_14th_Raw", axis=1)

# Save to a new Excel file
df.to_excel(output_file, index=False)
print(f"Saved updated DataFrame to: {output_file}")


Saved updated DataFrame to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V1_df.xlsx


In [6]:
import pandas as pd
import os

# File and folder paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V1_df.xlsx"
sheet_to_read = "Sheet1"
sheet_to_write = "Sheet2"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/V1_2nd_Raw"

# Load Sheet1
df = pd.read_excel(xlsx_file, sheet_name=sheet_to_read)

# Ensure FileName_V1_2nd is string type
df["FileName_V1_2nd"] = df["FileName_V1_2nd"].astype(str)

# Initialize validation column
df["V1_2nd_Raw_IsValid"] = "invalid"

# Create a set of valid names to match against
valid_names = set(df["FileName_V1_2nd"])

# Check each CSV file
for filename in os.listdir(csv_folder):
    if filename.endswith(".csv"):
        file_id = filename.replace(".csv", "")
        if file_id in valid_names:
            file_path = os.path.join(csv_folder, filename)
            try:
                csv_data = pd.read_csv(file_path)
                # Check if any row has 5 or more non-null entries
                if (csv_data.notna().sum(axis=1) >= 5).any():
                    df.loc[df["FileName_V1_2nd"] == file_id, "V1_2nd_Raw_IsValid"] = "valid"
            except Exception as e:
                print(f"Error reading {filename}: {e}")

# Save updated DataFrame to Sheet2
with pd.ExcelWriter(xlsx_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_to_write, index=False)

print("Sheet2 saved with V1_2nd_Raw_IsValid flags.")


Sheet2 saved with V1_2nd_Raw_IsValid flags.


In [7]:
import pandas as pd
import os

# File and folder paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V1_df.xlsx"
sheet_to_read = "Sheet2"
sheet_to_write = "Sheet3"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/V1_3rd_Raw"

# Load Sheet1
df = pd.read_excel(xlsx_file, sheet_name=sheet_to_read)

# Ensure FileName_V1_3rd is string type
df["FileName_V1_3rd"] = df["FileName_V1_3rd"].astype(str)

# Initialize validation column
df["V1_3rd_Raw_IsValid"] = "invalid"

# Create a set of valid names to match against
valid_names = set(df["FileName_V1_3rd"])

# Check each CSV file
for filename in os.listdir(csv_folder):
    if filename.endswith(".csv"):
        file_id = filename.replace(".csv", "")
        if file_id in valid_names:
            file_path = os.path.join(csv_folder, filename)
            try:
                csv_data = pd.read_csv(file_path)
                # Check if any row has 5 or more non-null entries
                if (csv_data.notna().sum(axis=1) >= 5).any():
                    df.loc[df["FileName_V1_3rd"] == file_id, "V1_3rd_Raw_IsValid"] = "valid"
            except Exception as e:
                print(f"Error reading {filename}: {e}")

# Save updated DataFrame to Sheet2
with pd.ExcelWriter(xlsx_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_to_write, index=False)

print("Sheet3 saved with V1_3rd_Raw_IsValid flags.")


Sheet3 saved with V1_3rd_Raw_IsValid flags.


In [8]:
import pandas as pd
import os

# File and folder paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V1_df.xlsx"
sheet_to_read = "Sheet3"
sheet_to_write = "Sheet4"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/V1_13th_Raw"

# Load Sheet1
df = pd.read_excel(xlsx_file, sheet_name=sheet_to_read)

# Ensure FileName_V1_13th is string type
df["FileName_V1_13th"] = df["FileName_V1_13th"].astype(str)

# Initialize validation column
df["V1_13th_Raw_IsValid"] = "invalid"

# Create a set of valid names to match against
valid_names = set(df["FileName_V1_13th"])

# Check each CSV file
for filename in os.listdir(csv_folder):
    if filename.endswith(".csv"):
        file_id = filename.replace(".csv", "")
        if file_id in valid_names:
            file_path = os.path.join(csv_folder, filename)
            try:
                csv_data = pd.read_csv(file_path)
                # Check if any row has 5 or more non-null entries
                if (csv_data.notna().sum(axis=1) >= 5).any():
                    df.loc[df["FileName_V1_13th"] == file_id, "V1_13th_Raw_IsValid"] = "valid"
            except Exception as e:
                print(f"Error reading {filename}: {e}")

# Save updated DataFrame to Sheet2
with pd.ExcelWriter(xlsx_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_to_write, index=False)

print("Sheet4 saved with V1_13th_Raw_IsValid flags.")


Sheet4 saved with V1_13th_Raw_IsValid flags.


In [9]:
import pandas as pd
import os

# File and folder paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V1_df.xlsx"
sheet_to_read = "Sheet4"
sheet_to_write = "Sheet5"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/V1_14th_Raw"

# Load Sheet1
df = pd.read_excel(xlsx_file, sheet_name=sheet_to_read)

# Ensure FileName_V1_14th is string type
df["FileName_V1_14th"] = df["FileName_V1_14th"].astype(str)

# Initialize validation column
df["V1_14th_Raw_IsValid"] = "invalid"

# Create a set of valid names to match against
valid_names = set(df["FileName_V1_14th"])

# Check each CSV file
for filename in os.listdir(csv_folder):
    if filename.endswith(".csv"):
        file_id = filename.replace(".csv", "")
        if file_id in valid_names:
            file_path = os.path.join(csv_folder, filename)
            try:
                csv_data = pd.read_csv(file_path)
                # Check if any row has 5 or more non-null entries
                if (csv_data.notna().sum(axis=1) >= 5).any():
                    df.loc[df["FileName_V1_14th"] == file_id, "V1_14th_Raw_IsValid"] = "valid"
            except Exception as e:
                print(f"Error reading {filename}: {e}")

# Save updated DataFrame to Sheet2
with pd.ExcelWriter(xlsx_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_to_write, index=False)

print("Sheet5 saved with V1_14th_Raw_IsValid flags.")


Sheet5 saved with V1_14th_Raw_IsValid flags.


### PARDS Risk V2 (1st / 12th, 2nd / 13th, 3rd / 14th) merged to PARDS_Risk_V2_df

In [4]:
import pandas as pd

# File and sheet setup
source_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"
sheet_to_read = "Sheet1"
sheet_to_write = "Sheet2"

# Load Sheet1
df = pd.read_excel(source_file, sheet_name=sheet_to_read)

# Format MRN as 9-digit string
df["MRN"] = df["MRN"].astype(str).str.zfill(9)

# Convert to datetime
df["1st_Time_Start"] = pd.to_datetime(df["1st_Time_Start"], errors='coerce')

# Filter rows where 1st_Time_Start is not null
df = df[df["1st_Time_Start"].notna()].copy()

# Generate additional time columns
df["2nd_Time_Start"] = df["1st_Time_Start"] + pd.Timedelta(hours=1)
df["3rd_Time_Start"] = df["1st_Time_Start"] + pd.Timedelta(hours=2)
df["12th_Time_Start"] = df["1st_Time_Start"] + pd.Timedelta(hours=12)
df["13th_Time_Start"] = df["12th_Time_Start"] + pd.Timedelta(hours=1)
df["14th_Time_Start"] = df["12th_Time_Start"] + pd.Timedelta(hours=2)

# Time formatting helper
def format_time(ts):
    return ts.strftime("%Y%m%d_%H") if pd.notnull(ts) else ""

# Filename columns
df["FileName_V2_1st"] = df.apply(lambda row: f"{row['PID']}_{format_time(row['1st_Time_Start'])}_V2_1st_Raw", axis=1)
df["FileName_V2_2nd"] = df.apply(lambda row: f"{row['PID']}_{format_time(row['2nd_Time_Start'])}_V2_2nd_Raw", axis=1)
df["FileName_V2_3rd"] = df.apply(lambda row: f"{row['PID']}_{format_time(row['3rd_Time_Start'])}_V2_3rd_Raw", axis=1)
df["FileName_V2_12th"] = df.apply(lambda row: f"{row['PID']}_{format_time(row['12th_Time_Start'])}_V2_12th_Raw", axis=1)
df["FileName_V2_13th"] = df.apply(lambda row: f"{row['PID']}_{format_time(row['13th_Time_Start'])}_V2_13th_Raw", axis=1)
df["FileName_V2_14th"] = df.apply(lambda row: f"{row['PID']}_{format_time(row['14th_Time_Start'])}_V2_14th_Raw", axis=1)

# Save to Sheet2
with pd.ExcelWriter(source_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_to_write, index=False)

print("Filtered and saved to Sheet2 with 9-digit MRN, time columns, and filename columns.")


Filtered and saved to Sheet2 with 9-digit MRN, time columns, and filename columns.


In [10]:
import pandas as pd
import os

# File and folder paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"
sheet_to_read = "Sheet2"
sheet_to_write = "Sheet3"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/V2_1st_Raw"

# Load Sheet1
df = pd.read_excel(xlsx_file, sheet_name=sheet_to_read)

# Ensure FileName_V2_1st is string type
df["FileName_V2_1st"] = df["FileName_V2_1st"].astype(str)

# Initialize validation column
df["V2_1st_Raw_IsValid"] = "invalid"

# Create a set of valid names to match against
valid_names = set(df["FileName_V2_1st"])

# Check each CSV file
for filename in os.listdir(csv_folder):
    if filename.endswith(".csv"):
        file_id = filename.replace(".csv", "")
        if file_id in valid_names:
            file_path = os.path.join(csv_folder, filename)
            try:
                csv_data = pd.read_csv(file_path)
                # Check if any row has 5 or more non-null entries
                if (csv_data.notna().sum(axis=1) >= 5).any():
                    df.loc[df["FileName_V2_1st"] == file_id, "V2_1st_Raw_IsValid"] = "valid"
            except Exception as e:
                print(f"Error reading {filename}: {e}")

# Save updated DataFrame to Sheet2
with pd.ExcelWriter(xlsx_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_to_write, index=False)

print("Sheet3 saved with V2_1st_Raw_IsValid flags.")


Sheet3 saved with V2_1st_Raw_IsValid flags.


In [11]:
import pandas as pd
import os

# File and folder paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"
sheet_to_read = "Sheet3"
sheet_to_write = "Sheet4"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/V2_2nd_Raw"

# Load Sheet1
df = pd.read_excel(xlsx_file, sheet_name=sheet_to_read)

# Ensure FileName_V2_2nd is string type
df["FileName_V2_2nd"] = df["FileName_V2_2nd"].astype(str)

# Initialize validation column
df["V2_2nd_Raw_IsValid"] = "invalid"

# Create a set of valid names to match against
valid_names = set(df["FileName_V2_2nd"])

# Check each CSV file
for filename in os.listdir(csv_folder):
    if filename.endswith(".csv"):
        file_id = filename.replace(".csv", "")
        if file_id in valid_names:
            file_path = os.path.join(csv_folder, filename)
            try:
                csv_data = pd.read_csv(file_path)
                # Check if any row has 5 or more non-null entries
                if (csv_data.notna().sum(axis=1) >= 5).any():
                    df.loc[df["FileName_V2_2nd"] == file_id, "V2_2nd_Raw_IsValid"] = "valid"
            except Exception as e:
                print(f"Error reading {filename}: {e}")

# Save updated DataFrame to Sheet2
with pd.ExcelWriter(xlsx_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_to_write, index=False)

print("Sheet4 saved with V2_2nd_Raw_IsValid flags.")


Sheet4 saved with V2_2nd_Raw_IsValid flags.


In [12]:
import pandas as pd
import os

# File and folder paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"
sheet_to_read = "Sheet4"
sheet_to_write = "Sheet5"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/V2_3rd_Raw"

# Load Sheet1
df = pd.read_excel(xlsx_file, sheet_name=sheet_to_read)

# Ensure FileName_V2_3rd is string type
df["FileName_V2_3rd"] = df["FileName_V2_3rd"].astype(str)

# Initialize validation column
df["V2_3rd_Raw_IsValid"] = "invalid"

# Create a set of valid names to match against
valid_names = set(df["FileName_V2_3rd"])

# Check each CSV file
for filename in os.listdir(csv_folder):
    if filename.endswith(".csv"):
        file_id = filename.replace(".csv", "")
        if file_id in valid_names:
            file_path = os.path.join(csv_folder, filename)
            try:
                csv_data = pd.read_csv(file_path)
                # Check if any row has 5 or more non-null entries
                if (csv_data.notna().sum(axis=1) >= 5).any():
                    df.loc[df["FileName_V2_3rd"] == file_id, "V2_3rd_Raw_IsValid"] = "valid"
            except Exception as e:
                print(f"Error reading {filename}: {e}")

# Save updated DataFrame to Sheet2
with pd.ExcelWriter(xlsx_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_to_write, index=False)

print("Sheet5 saved with V2_3rd_Raw_IsValid flags.")


Sheet5 saved with V2_3rd_Raw_IsValid flags.


In [13]:
import pandas as pd
import os

# File and folder paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"
sheet_to_read = "Sheet5"
sheet_to_write = "Sheet6"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/V2_12th_Raw"

# Load Sheet1
df = pd.read_excel(xlsx_file, sheet_name=sheet_to_read)

# Ensure FileName_V2_12th is string type
df["FileName_V2_12th"] = df["FileName_V2_12th"].astype(str)

# Initialize validation column
df["V2_12th_Raw_IsValid"] = "invalid"

# Create a set of valid names to match against
valid_names = set(df["FileName_V2_12th"])

# Check each CSV file
for filename in os.listdir(csv_folder):
    if filename.endswith(".csv"):
        file_id = filename.replace(".csv", "")
        if file_id in valid_names:
            file_path = os.path.join(csv_folder, filename)
            try:
                csv_data = pd.read_csv(file_path)
                # Check if any row has 5 or more non-null entries
                if (csv_data.notna().sum(axis=1) >= 5).any():
                    df.loc[df["FileName_V2_12th"] == file_id, "V2_12th_Raw_IsValid"] = "valid"
            except Exception as e:
                print(f"Error reading {filename}: {e}")

# Save updated DataFrame to Sheet2
with pd.ExcelWriter(xlsx_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_to_write, index=False)

print("Sheet6 saved with V2_12th_Raw_IsValid flags.")


Sheet6 saved with V2_12th_Raw_IsValid flags.


In [14]:
import pandas as pd
import os

# File and folder paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"
sheet_to_read = "Sheet6"
sheet_to_write = "Sheet7"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/V2_13th_Raw"

# Load Sheet1
df = pd.read_excel(xlsx_file, sheet_name=sheet_to_read)

# Ensure FileName_V2_13th is string type
df["FileName_V2_13th"] = df["FileName_V2_13th"].astype(str)

# Initialize validation column
df["V2_13th_Raw_IsValid"] = "invalid"

# Create a set of valid names to match against
valid_names = set(df["FileName_V2_13th"])

# Check each CSV file
for filename in os.listdir(csv_folder):
    if filename.endswith(".csv"):
        file_id = filename.replace(".csv", "")
        if file_id in valid_names:
            file_path = os.path.join(csv_folder, filename)
            try:
                csv_data = pd.read_csv(file_path)
                # Check if any row has 5 or more non-null entries
                if (csv_data.notna().sum(axis=1) >= 5).any():
                    df.loc[df["FileName_V2_13th"] == file_id, "V2_13th_Raw_IsValid"] = "valid"
            except Exception as e:
                print(f"Error reading {filename}: {e}")

# Save updated DataFrame to Sheet2
with pd.ExcelWriter(xlsx_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_to_write, index=False)

print("Sheet7 saved with V2_13th_Raw_IsValid flags.")


Sheet7 saved with V2_13th_Raw_IsValid flags.


In [15]:
import pandas as pd
import os

# File and folder paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"
sheet_to_read = "Sheet7"
sheet_to_write = "Sheet8"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/V2_14th_Raw"

# Load Sheet1
df = pd.read_excel(xlsx_file, sheet_name=sheet_to_read)

# Ensure FileName_V2_14th is string type
df["FileName_V2_14th"] = df["FileName_V2_14th"].astype(str)

# Initialize validation column
df["V2_14th_Raw_IsValid"] = "invalid"

# Create a set of valid names to match against
valid_names = set(df["FileName_V2_14th"])

# Check each CSV file
for filename in os.listdir(csv_folder):
    if filename.endswith(".csv"):
        file_id = filename.replace(".csv", "")
        if file_id in valid_names:
            file_path = os.path.join(csv_folder, filename)
            try:
                csv_data = pd.read_csv(file_path)
                # Check if any row has 5 or more non-null entries
                if (csv_data.notna().sum(axis=1) >= 5).any():
                    df.loc[df["FileName_V2_14th"] == file_id, "V2_14th_Raw_IsValid"] = "valid"
            except Exception as e:
                print(f"Error reading {filename}: {e}")

# Save updated DataFrame to Sheet2
with pd.ExcelWriter(xlsx_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_to_write, index=False)

print("Sheet8 saved with V2_14th_Raw_IsValid flags.")


Sheet8 saved with V2_14th_Raw_IsValid flags.


### Check duplicates within V2_df and overlap between V1_df and V2_df

In [17]:
import pandas as pd

# File and sheet setup
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"
sheet_to_read = "Sheet8"
sheet_to_write = "Sheet9"

# Load Sheet8
df = pd.read_excel(xlsx_file, sheet_name=sheet_to_read)

# Check for duplicates in 'FileName_V2_1st'
df["V2_1st_IsDuplicate"] = df.duplicated(subset=["FileName_V2_1st"], keep=False).map({True: "yes", False: "no"})

# Save to Sheet9 in the same Excel file
with pd.ExcelWriter(xlsx_file, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df.to_excel(writer, sheet_name=sheet_to_write, index=False)

print(f"Marked duplicates and saved updated DataFrame to {sheet_to_write}.")


Marked duplicates and saved updated DataFrame to Sheet9.


In [2]:
import pandas as pd

# File paths
v1_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V1_df.xlsx"
v2_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"

# Read the sheets
df_v1 = pd.read_excel(v1_file, sheet_name="Sheet6")
df_v2 = pd.read_excel(v2_file, sheet_name="Sheet9")

# Convert to datetime
df_v1["1st_Time_Start"] = pd.to_datetime(df_v1["1st_Time_Start"], errors='coerce')
df_v2["1st_Time_Start"] = pd.to_datetime(df_v2["1st_Time_Start"], errors='coerce')

# Find overlapping timestamps
overlapping_times = pd.merge(
    df_v1[["1st_Time_Start"]],
    df_v2[["1st_Time_Start"]],
    on="1st_Time_Start",
    how="inner"
)["1st_Time_Start"].dropna().unique()

# Mark overlapping rows
df_v1["Is_Overlapping"] = df_v1["1st_Time_Start"].isin(overlapping_times).map({True: "yes", False: "no"})
df_v2["Is_Overlapping"] = df_v2["1st_Time_Start"].isin(overlapping_times).map({True: "yes", False: "no"})

# Save updated dataframes to new sheets
with pd.ExcelWriter(v1_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df_v1.to_excel(writer, sheet_name="Sheet7", index=False)

with pd.ExcelWriter(v2_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df_v2.to_excel(writer, sheet_name="Sheet10", index=False)

print("Updated sheets saved: Sheet7 (V1) and Sheet10 (V2) with overlap info.")


Updated sheets saved: Sheet7 (V1) and Sheet10 (V2) with overlap info.
